In [0]:
from datetime import datetime
import mysql.connector
import pandas as pd
from pyspark.sql import SparkSession

from spark_session import create_spark_session

spark = create_spark_session()

jdbc_url = "jdbc:mysql://relational.fel.cvut.cz:3306/imdb_ijs"

hoje = datetime.now()

# Ajustando para caminho do DBFS no Databricks
output_base_path = f"/dbfs/mnt/landing/imdb_ijs/{hoje.year}/{hoje.month:02}/{hoje.day:02}"

print("JDBC URL:", jdbc_url)
print("Base de saída:", output_base_path)

conn = mysql.connector.connect(
    host="relational.fel.cvut.cz",
    port=3306,
    database="imdb_ijs",
    user="guest",
    password="ctu-relational",
)

query_tabelas = """
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'imdb_ijs'
    ORDER BY table_name
"""

cursor = conn.cursor()
cursor.execute(query_tabelas)
tabelas = [row[0] for row in cursor.fetchall()]
cursor.close()

df_tables = spark.createDataFrame(pd.DataFrame({"table_name": tabelas}))
df_tables.show()

for tabela in tabelas:
    df_pd = pd.read_sql(f"SELECT * FROM {tabela}", conn)

    if df_pd.empty:
        continue

    df_spark = spark.createDataFrame(df_pd)

    tabela_output_path = f"{output_base_path}/{tabela}"

    # Salvar em Delta, em vez de CSV
    df_spark.write.format("delta").mode("overwrite").saveAsTable(f"landing_{tabela}")

conn.close()
